In [ ]:
# 1. Clone the repository
!git clone https://github.com/SinaTabakhi/MAGNET.git
%cd MAGNET

# 2. Detect Colab's current PyTorch version dynamically
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
print(f"Colab is using Torch {torch_version} with CUDA {cuda_version}")

# 3. Install PyG binaries that match Colab's environment exactly
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html
!pip install torch-geometric==2.4.0

# 4. Install the remaining requirements
!pip install lightning==2.1.3 pandas matplotlib umap-learn yacs comet_ml "ray[tune]"

Cloning into 'MAGNET'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 202 (delta 8), reused 0 (delta 0), pack-reused 167 (from 1)
Receiving objects: 100% (202/202), 53.31 MiB | 14.83 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (151/151), done.
/content/MAGNET
Colab is using Torch 2.11.0 with CUDA 128
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 36.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 36.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
%cd /content/MAGNET

/content/MAGNET


In [ ]:
from google.colab import files

# This will prompt you to select files from your computer
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Successfully uploaded "{filename}" to the root directory.')

Saving BLCA_10_confidence_scores.csv to BLCA_10_confidence_scores.csv
Saving BLCA_20_confidence_scores.csv to BLCA_20_confidence_scores.csv
Saving BLCA_30_confidence_scores.csv to BLCA_30_confidence_scores.csv
Saving BLCA_40_confidence_scores.csv to BLCA_40_confidence_scores.csv
Saving BLCA_50_confidence_scores.csv to BLCA_50_confidence_scores.csv
Saving BRCA_10_confidence_scores.csv to BRCA_10_confidence_scores.csv
Saving BRCA_20_confidence_scores.csv to BRCA_20_confidence_scores.csv
Saving BRCA_30_confidence_scores.csv to BRCA_30_confidence_scores.csv
Saving BRCA_40_confidence_scores.csv to BRCA_40_confidence_scores.csv
Saving BRCA_50_confidence_scores.csv to BRCA_50_confidence_scores.csv
Saving OV_10_confidence_scores.csv to OV_10_confidence_scores.csv
Saving OV_20_confidence_scores.csv to OV_20_confidence_scores.csv
Saving OV_30_confidence_scores.csv to OV_30_confidence_scores.csv
Saving OV_40_confidence_scores.csv to OV_40_confidence_scores.csv
Saving OV_50_confidence_scores.csv t

In [ ]:
!python main_inference.py --cfg configs/MAGNET_OV.yaml

2026-07-23 11:27:55.088091: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

=== Starting Optimized Robustness Ablation Study on OV ===


[SEED 10] -> Initializing and Training Base Model...
[ConfidenceScoreManager] Successfully loaded 182 patient scores from OV_10_confidence_scores.csv
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
      [Progress] Epoch 20/200 completed.
      [Progress] Epoch 40/200 completed.
      [Progress] Epoch 60/200 completed.
      [Progress] Epoch 80/200 completed.
      [Progress] Epoch 100/200 completed.
      [Progress] Epoch 120/200 completed.
      [Progress] Epoch 140

In [ ]:
#manager class
import pandas as pd
import torch

class ConfidenceScoreManager:
    def __init__(self, num_modalities, target_modality_idx=0):
        """
        Manages external confidence scores for the MAGNET Ablation Study.

        Args:
            num_modalities (int): Total number of modalities in the model.
            target_modality_idx (int): The index of the modality that has missing/imputed
                                       data corresponding to the CSV scores.
        """
        self.num_modalities = num_modalities
        self.target_modality_idx = target_modality_idx
        self.scores_dict = {}

    def load_missing_rate(self, csv_filepath):
        """
        Loads the confidence scores for a specific missing rate.
        """
        df = pd.read_csv(csv_filepath)

        # Create a fast lookup dictionary mapping sample_id -> confidence_score
        self.scores_dict = dict(zip(df['sample_id'], df['confidence_score']))
        print(f"Loaded {len(self.scores_dict)} patient confidence scores from {csv_filepath}")

    def get_batch_tensor(self, batch_sample_ids, device='cpu'):
        """
        Generates the [batch_size, num_modalities] tensor for the current batch.
        """
        batch_size = len(batch_sample_ids)

        # Initialize all modalities with full confidence (1.0)
        ext_conf = torch.ones((batch_size, self.num_modalities), dtype=torch.float32, device=device)

        # Populate the specific target modality with scores from the CSV
        for i, sample_id in enumerate(batch_sample_ids):
            # Default to 1.0 if a sample_id is unexpectedly missing from the CSV
            score = self.scores_dict.get(sample_id, 1.0)
            ext_conf[i, self.target_modality_idx] = score

        return ext_conf

In [ ]:
# base_models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        return self.encoder_layers(x)


# -------------------------------------------------------------------------------
# Generalized fuse() function (Supports Ablation Stress Tests)
# -------------------------------------------------------------------------------
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None, external_confidence=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    elif mode == "external":
        assert external_confidence is not None, "external_confidence is required for 'external' mode"
        # external_confidence shape expected: [batch_size, num_modalities]
        ext_conf = external_confidence.unsqueeze(1).expand(-1, num_heads, -1)
        ext_conf = ext_conf * mask.unsqueeze(1)  # Respect patient mask
        # Normalize weights so they sum to 1.0 per head, acting like a standard attention distribution
        att_weights = ext_conf / ext_conf.sum(dim=-1, keepdim=True).clamp(min=1e-9)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', 'shared', or 'external'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert self.head_dims * num_heads == hid_dims, "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Shared learnable weights
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original", external_confidence=None):
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # 2. Call standalone fuse function dynamically based on mode argument
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights,
            external_confidence=external_confidence
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int, aggr="mean", bias: bool=True, edge_dim: int=None, **kwargs):
        super().__init__(aggr, **kwargs)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)
        self.act_msg = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr=None):
        if isinstance(x, Tensor):
            x = (x, x)
        x = (self.lin(x[0]).relu(), x[1])
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = out + self.lin_r(x[1])
        return out

    def message(self, x_j, edge_attr=None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList([EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1) for _ in range(num_layers)])
        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')

    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = F.dropout(F.relu(conv(x, edge_index, edge_attr=edge_attr)), p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        logits = self.final_layer(x)
        if return_embedding:
            return logits, x
        return logits

In [ ]:
#trainer.py
from typing import List
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import lightning as L

from model.base_models import MLPEncoder, MultiHeadAttentionLayer, GNNDecoder
from dataloader.dataloader import MultiomicsDataset
from utils import (
    align_modalities,
    evaluate_classification_performance,
    generate_training_confidences,
    stress_test_a_perturbation,
    stress_test_b_perturbation,
    ConfidenceScoreManager
)


class MAGNETTrainer(L.LightningModule):
    def __init__(
        self,
        dataset: MultiomicsDataset,
        unimodal_encoders: List[MLPEncoder],
        attention_layer: MultiHeadAttentionLayer,
        gnn_decoder: GNNDecoder,
        loss_fn: CrossEntropyLoss,
        lr: float,
        wd: float,
        fusion_mode: str = "original",
        target_modality: int = None,
        gamma: float = 1.0,
        stress_test_mode: str = "control",
        confidence_manager: ConfidenceScoreManager = None
    ):
        super().__init__()
        self.save_hyperparameters("lr", "wd", "fusion_mode", "target_modality", "gamma", "stress_test_mode")
        self.dataset = dataset
        self.unimodal_encoders = nn.ModuleList(unimodal_encoders)
        self.attention_layer = attention_layer
        self.gnn_decoder = gnn_decoder
        self.loss_fn = loss_fn
        self.lr = lr
        self.wd = wd

        self.fusion_mode = fusion_mode
        self.target_modality = target_modality
        self.gamma = gamma
        self.stress_test_mode = stress_test_mode
        self.confidence_manager = confidence_manager

    def configure_optimizers(self):
        optimizer = Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = StepLR(optimizer, step_size=20, gamma=0.8)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}

    def similarity_kl_loss(self, similarities, embeddings, mask=None, alpha=1.0, eps=1e-12):
        device = embeddings.device
        n = embeddings.size(0)
        sum_sq = torch.sum(embeddings ** 2, dim=1)
        dist_sq = sum_sq.unsqueeze(1) + sum_sq.unsqueeze(0) - 2 * torch.matmul(embeddings, embeddings.T)
        q_matrix = torch.pow(1 + dist_sq / alpha, -(alpha + 1) / 2) * (1 - torch.eye(n, device=device))

        if mask is not None:
            q_matrix = q_matrix.masked_fill(mask == 0, 0.0)
            similarities = similarities.masked_fill(mask == 0, 0.0)

        q_distribution = torch.clamp(q_matrix / (q_matrix.sum() + eps), min=eps)
        p_distribution = torch.clamp(similarities / (similarities.sum() + eps), min=eps)
        return torch.sum(p_distribution * torch.log(p_distribution / q_distribution))

    def forward(self, x, mode, return_embedding=False):
        data, data_indices, modality_mask, labels = x
        encoded_modalities = [encoder(data[i]) for i, encoder in enumerate(self.unimodal_encoders)]

        aligned_modalities = align_modalities(encoded_modalities, data_indices, labels)
        modality_embeddings = torch.stack(aligned_modalities, dim=1)
        batch_size = modality_embeddings.size(0)
        num_modalities = self.dataset.num_omics
        sample_ids = list(labels.index)

        # -----------------------------------------------------------------
        # MASK OVERRIDE & CONFIDENCE EXTRACTION FROM CSV
        # -----------------------------------------------------------------
        # Override outdated mask: missing data has been imputed, so mark all as present (1.0)
        active_mask = torch.ones_like(modality_mask)

        if mode == "train":
            if self.confidence_manager is not None:
                external_confidence = self.confidence_manager.get_batch_tensor(sample_ids, device=self.device)
            else:
                external_confidence = generate_training_confidences(batch_size, num_modalities, device=self.device)
            current_fusion_mode = "external" if self.fusion_mode == "external" else self.fusion_mode
        else:
            if self.fusion_mode == "external":
                if self.confidence_manager is not None:
                    external_confidence = self.confidence_manager.get_batch_tensor(sample_ids, device=self.device)
                else:
                    external_confidence = torch.full((batch_size, num_modalities), 0.9, device=self.device)
                    if self.target_modality is not None:
                        external_confidence[:, self.target_modality] = self.gamma

                # Dynamically apply evaluation stress test modifications
                if self.stress_test_mode == "test_a":
                    external_confidence = stress_test_a_perturbation(external_confidence, drop_ratio=0.2, drop_val=0.1)
                elif self.stress_test_mode == "test_b":
                    modality_embeddings, external_confidence = stress_test_b_perturbation(
                        modality_embeddings, external_confidence, drop_ratio=0.2, high_conf_val=0.9
                    )
                current_fusion_mode = "external"
            else:
                external_confidence = None
                current_fusion_mode = self.fusion_mode

        fused_embeddings, _ = self.attention_layer(
            modality_embeddings,
            active_mask,
            mode=current_fusion_mode,
            external_confidence=external_confidence
        )

        graph_data, similarities, edge_mask = self.dataset.build_graph_data(fused_embeddings, mode)

        if return_embedding:
            output, gnn_embeddings = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr, return_embedding=True)
            return aligned_modalities, fused_embeddings, gnn_embeddings, graph_data

        output = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
        return output, graph_data, similarities, fused_embeddings, edge_mask

    def training_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="train")
        train_kl_loss = self.similarity_kl_loss(similarities, fused_embeddings, edge_mask)

        mask = graph_data.train_mask
        y_true, y_logit = graph_data.y.squeeze()[mask], output[mask]
        train_cls_loss = self.loss_fn(y_logit, y_true)
        total_loss = train_cls_loss + 0.1 * train_kl_loss

        self.log('train_kl_loss', train_kl_loss.detach().cpu(), prog_bar=False)
        self.log('train_cls_loss', train_cls_loss.detach().cpu(), prog_bar=False)
        self.log('train_total_loss', total_loss.detach().cpu(), prog_bar=False)
        return total_loss

    def test_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="test")
        mask = graph_data.test_mask
        y_true, y_logit = graph_data.y.squeeze()[mask], output[mask]

        probs = torch.softmax(y_logit, dim=-1)
        metrics = evaluate_classification_performance(y_true, probs, self.dataset.num_classes)

        for name, value in metrics.items():
            self.log(f'test_{name}', value, prog_bar=False)

    def _custom_data_loader(self): return self.dataset
    def train_dataloader(self): return self._custom_data_loader()
    def test_dataloader(self): return self._custom_data_loader()

In [ ]:
#main_inference.py
import os
import gc
import warnings
import argparse
import pandas as pd
import numpy as np
import lightning as L
from lightning.pytorch.callbacks import Callback
from torch.nn import CrossEntropyLoss
import torch

from utils import seed_everything, ConfidenceScoreManager
from dataloader.dataloader import MultiomicsDataset
from model.base_models import MLPEncoder, GNNDecoder, MultiHeadAttentionLayer
from trainer.trainer import MAGNETTrainer
from configs.config import get_cfg_defaults

class PrintEpochCallback(Callback):
    """Prints training progress strictly 1/10th of the max epochs to reduce clutter."""
    def on_train_epoch_end(self, trainer, pl_module):
        interval = max(1, trainer.max_epochs // 10)
        if (trainer.current_epoch + 1) % interval == 0:
            print(f"      [Progress] Epoch {trainer.current_epoch + 1}/{trainer.max_epochs} completed.")

def arg_parse():
    parser = argparse.ArgumentParser(description="MAGNET Optimized Stress Test Ablation Study")
    parser.add_argument("--cfg", required=True, help="path to config file", type=str)
    return parser.parse_args()

def train_base_model(cfg, split_folder, seed, lightning_dir, target_modality_idx=1):
    """Initializes and trains the base model ONCE for a given seed using CSV confidences."""
    seed_everything(seed)

    # Load Data
    multiomics = MultiomicsDataset(
        split_folder=split_folder, dataset_name=cfg.DATASET.NAME,
        modalities=cfg.DATASET.OMICS, seed=seed,
        num_classes=cfg.DATASET.NUM_CLASSES, class_names=cfg.DATASET.CLASS_NAMES,
        sparsity_rate=cfg.DATASET.SPARSITY_RATES, tune_hyperparameters=False
    )

    modality_features = [multiomics.get_data(omics_idx=i).shape[1] for i in range(len(cfg.DATASET.OMICS))]

    # Setup Confidence Manager & Load CSV for the seed
    num_modalities = len(cfg.DATASET.OMICS)
    confidence_manager = ConfidenceScoreManager(num_modalities=num_modalities, target_modality_idx=target_modality_idx)

    csv_filename = f"{cfg.DATASET.NAME}_{seed}_confidence_scores.csv"
    if os.path.exists(csv_filename):
        confidence_manager.load_csv(csv_filename)
    elif os.path.exists(os.path.join(split_folder, csv_filename)):
        confidence_manager.load_csv(os.path.join(split_folder, csv_filename))
    else:
        print(f"[Warning] CSV {csv_filename} not found. Defaulting confidence scores to 1.0.")

    # Build Models
    encoders = [
        MLPEncoder(in_dims=modality_features[i], hid_dims=cfg.ENCODER.HID_DIMS, dropout_rate=cfg.ENCODER.DROPOUT_RATE)
        for i in range(len(cfg.DATASET.OMICS))
    ]
    attention_layer = MultiHeadAttentionLayer(hid_dims=cfg.ENCODER.HID_DIMS, num_heads=cfg.ATTENTION.NUM_HEADS, num_modalities=num_modalities)
    decoder = GNNDecoder(hid_dims=cfg.DECODER.HID_DIMS, out_dims=cfg.DATASET.NUM_CLASSES, num_layers=cfg.DECODER.NUM_LAYERS, dropout_rate=cfg.DECODER.DROPOUT_RATE, negative_slope=cfg.DECODER.NEGATIVE_SLOPE)

    # Setup Trainer
    model = MAGNETTrainer(
        dataset=multiomics, unimodal_encoders=encoders,
        attention_layer=attention_layer, gnn_decoder=decoder,
        loss_fn=CrossEntropyLoss(), lr=cfg.SOLVER.LR, wd=cfg.SOLVER.WD,
        fusion_mode="external", target_modality=target_modality_idx, gamma=1.0,
        stress_test_mode="control", confidence_manager=confidence_manager
    )

    trainer = L.Trainer(
        max_epochs=cfg.SOLVER.MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        callbacks=[PrintEpochCallback()],
        default_root_dir=lightning_dir
    )

    trainer.fit(model)
    return model, trainer, multiomics

def main():
    warnings.filterwarnings(action="ignore")
    args = arg_parse()
    cfg = get_cfg_defaults()
    cfg.merge_from_file(args.cfg)
    cfg.freeze()

    lightning_dir = os.path.join(cfg.RESULT.OUTPUT_DIR, cfg.RESULT.LIGHTNING_LOG_DIR)
    if not os.path.exists(lightning_dir):
        os.makedirs(lightning_dir)
    split_folder = os.path.join(cfg.DATASET.ROOT, cfg.DATASET.SPLITS)

    seeds = [10, 20, 30, 40, 50]
    stress_conditions = ["control", "test_a", "test_b"]
    condition_labels = {
        "control": "Unperturbed Control",
        "test_a": "Stress Test A (False Low Conf)",
        "test_b": "Stress Test B (Noise + High Conf)"
    }

    # Dictionary to aggregate results per condition
    results_dict = {cond: {"acc": [], "f1": []} for cond in stress_conditions}

    print(f"\n{'='*60}")
    print(f"=== Starting Optimized Robustness Ablation Study on {cfg.DATASET.NAME} ===")
    print(f"{'='*60}\n")

    # OPTIMIZED LOOP: Train once per seed, evaluate across all conditions
    for seed in seeds:
        print(f"\n[SEED {seed}] -> Initializing and Training Base Model...")
        model, trainer, dataset = train_base_model(cfg, split_folder, seed, lightning_dir, target_modality_idx=1)

        for condition in stress_conditions:
            print(f"    -> Evaluating Condition: {condition_labels[condition]}")

            # Switch the evaluation mode on the same trained model
            model.stress_test_mode = condition
            trainer.test(model, verbose=False)

            metrics = trainer.logged_metrics
            acc_key = next((k for k in metrics.keys() if 'acc' in k.lower()), None)
            f1_key = next((k for k in metrics.keys() if 'f1' in k.lower()), None)

            acc = metrics[acc_key].item() if acc_key else 0.0
            f1 = metrics[f1_key].item() if f1_key else 0.0

            results_dict[condition]["acc"].append(acc)
            results_dict[condition]["f1"].append(f1)

        # Garbage Collection (Clean up before the next seed)
        del model, trainer, dataset
        gc.collect()
        torch.cuda.empty_cache()

    # Process and format the aggregated results
    stress_test_results = []
    for condition in stress_conditions:
        acc_list = results_dict[condition]["acc"]
        f1_list = results_dict[condition]["f1"]

        mean_acc, std_acc = np.mean(acc_list), np.std(acc_list)
        mean_f1, std_f1 = np.mean(f1_list), np.std(f1_list)

        stress_test_results.append({
            "Condition": condition_labels[condition],
            "Accuracy": f"{mean_acc:.4f} ± {std_acc:.4f}",
            "Macro-F1": f"{mean_f1:.4f} ± {std_f1:.4f}"
        })

    # Terminal Output & Saving
    print(f"\n\n{'='*60}")
    print(f"=== FINAL DELIVERABLE: {cfg.DATASET.NAME} STRUCTURAL STRESS TEST ===")
    print(f"{'='*60}\n")

    df_stress_test = pd.DataFrame(stress_test_results)
    print(df_stress_test.to_string(index=False))
    print("\n")

    output_csv = os.path.join(cfg.RESULT.OUTPUT_DIR, f"{cfg.DATASET.NAME}_confidence_stress_test.csv")
    df_stress_test.to_csv(output_csv, index=False)
    print(f"Results successfully saved to:\n - {output_csv}")

if __name__ == '__main__':
    main()

In [ ]:
#utils.py
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import csv
import random

import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef
from sklearn.metrics import silhouette_score, davies_bouldin_score
from lightning.pytorch.loggers import CometLogger, CSVLogger
import umap
import matplotlib.pyplot as plt


# -----------------------------------------------------------------
# CONFIDENCE SCORE MANAGER CLASS
# -----------------------------------------------------------------
class ConfidenceScoreManager:
    def __init__(self, num_modalities: int, target_modality_idx: int = 1):
        """
        Manages external confidence scores loaded from CSV for MAGNET.

        Args:
            num_modalities (int): Total number of modalities in the model.
            target_modality_idx (int): The index of the imputed target modality.
        """
        self.num_modalities = num_modalities
        self.target_modality_idx = target_modality_idx
        self.scores_dict = {}

    def load_csv(self, csv_filepath: str):
        """Loads patient confidence scores from CSV file."""
        if os.path.exists(csv_filepath):
            df = pd.read_csv(csv_filepath)
            self.scores_dict = dict(zip(df['sample_id'], df['confidence_score']))
            print(f"[ConfidenceScoreManager] Successfully loaded {len(self.scores_dict)} patient scores from {csv_filepath}")
        else:
            print(f"[ConfidenceScoreManager] Warning: File {csv_filepath} not found. Defaulting missing scores to 1.0.")
            self.scores_dict = {}

    def get_batch_tensor(self, batch_sample_ids: list, device='cpu') -> torch.Tensor:
        """
        Generates the [batch_size, num_modalities] external confidence tensor
        based on the patient IDs of the current batch.
        """
        batch_size = len(batch_sample_ids)
        ext_conf = torch.ones((batch_size, self.num_modalities), dtype=torch.float32, device=device)

        if self.scores_dict:
            for i, sample_id in enumerate(batch_sample_ids):
                score = self.scores_dict.get(sample_id, 1.0)
                ext_conf[i, self.target_modality_idx] = score

        return ext_conf


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def create_file(file_dir, header):
    with open(file_dir, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)


def save_output(file_dir, output, multi_rows=False):
    with open(file_dir, mode='a', newline='') as file:
        writer = csv.writer(file)
        if multi_rows:
            writer.writerows(output)
        else:
            writer.writerow(output)


def sort_file(origin_dir, sorted_dir, by):
    df = pd.read_csv(origin_dir)
    df = df.sort_values(by=by)
    df.to_csv(sorted_dir, index=False)


def load_label(dataset_folder: str, split_type: str):
    train_labels_path = os.path.join(dataset_folder, f"label_train_{split_type}.csv")
    val_labels_path = os.path.join(dataset_folder, f"label_val_{split_type}.csv")
    test_labels_path = os.path.join(dataset_folder, f"label_test_{split_type}.csv")

    train_labels = pd.read_csv(train_labels_path, index_col=0)
    val_labels = pd.read_csv(val_labels_path, index_col=0)
    test_labels = pd.read_csv(test_labels_path, index_col=0)

    for df in [train_labels, val_labels, test_labels]:
        if df.empty and "Class" in df.columns:
            df["Class"] = df["Class"].astype("int64")

    return train_labels, val_labels, test_labels


def load_dataset(dataset_folder: str, modality_name: str, labels: pd.DataFrame):
    unimodal_path = os.path.join(dataset_folder, f"{modality_name}.csv")
    unimodal_data = pd.read_csv(unimodal_path, index_col=0)

    common_indices = labels.index.intersection(unimodal_data.index)
    unimodal_data = unimodal_data.loc[common_indices].sort_index()

    return unimodal_data


def align_modalities(data, data_indices, label):
    aligned_modalities = []
    for modality_idx, data_modality in enumerate(data):
        modality_indices = data_indices[modality_idx]

        aligned_data = torch.zeros((len(label.index), data_modality.shape[1]),
                                   dtype=data_modality.dtype, device=data_modality.device)
        index_mapping = {patient_id: idx for idx, patient_id in enumerate(modality_indices)}

        for patient_row, patient_id in enumerate(label.index):
            if patient_id in index_mapping:
                aligned_data[patient_row] = data_modality[index_mapping[patient_id]]

        aligned_modalities.append(aligned_data)

    return aligned_modalities


def cosine_distance(x1, x2, mask=None, eps=1e-8):
    if mask is not None:
        x1 = x1 * mask
        x2 = x2 * mask

    norm_i = torch.norm(x1, p=2)
    norm_j = torch.norm(x2, p=2)
    similarity = torch.dot(x1, x2) / (norm_i * norm_j + eps)
    return similarity


def evaluate_classification_performance(y_true, probs, num_classes):
    y_pred = torch.argmax(probs, dim=-1)
    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()

    acc = accuracy_score(y_true_np, y_pred_np)
    macro_f1 = f1_score(y_true_np, y_pred_np, average="macro", zero_division=0)

    metrics = {
        'Accuracy': torch.tensor(acc, dtype=torch.float32),
        'Macro_F1': torch.tensor(macro_f1, dtype=torch.float32)
    }
    return metrics


def create_logger(config, name, style, use, support=True, hyper_params=None):
    if style == "comet":
        if use and support:
            save_dir = os.path.join(config.RESULT.OUTPUT_DIR, config.COMET.LOG_DIR)
            comet_logger = CometLogger(
                project_name=config.COMET.PROJECT_NAME,
                workspace=config.COMET.WORKSPACE,
                save_dir=save_dir,
                experiment_name=f"{name}",
                auto_output_logging="simple",
                log_graph=True,
                log_code=False,
                log_git_metadata=False,
                log_git_patch=False,
                auto_param_logging=False,
                auto_metric_logging=False
            )
            if hyper_params is None:
                hyper_params = {
                    "sparsity_rate": config.DATASET.SPARSITY_RATES,
                    "hid_dims": config.ENCODER.HID_DIMS,
                    "mlp_dropout_rate": config.ENCODER.DROPOUT_RATE,
                    "gat_num_layers": config.DECODER.NUM_LAYERS,
                    "gat_dropout_rate": config.DECODER.DROPOUT_RATE,
                    "gat_negative_slope": config.DECODER.NEGATIVE_SLOPE,
                    "lr": config.SOLVER.LR,
                    "wd": config.SOLVER.WD,
                    "max_epochs": config.SOLVER.MAX_EPOCHS
                }
            comet_logger.experiment.log_parameters(hyper_params)
            return comet_logger
    elif style == "csv":
        if use:
            save_dir = os.path.join(config.RESULT.OUTPUT_DIR, config.CSV.LOG_DIR)
            csv_logger = CSVLogger(save_dir=save_dir, name=f"{name}")
            return csv_logger
    return None


def plot_umap(config, embeddings_to_plot, labels_to_plot, class_names, save_path, seed):
    all_values = np.concatenate([np.array(values) for values in labels_to_plot.values()])
    classes = np.unique(all_values)

    if len(class_names) == 2:
        colormap = plt.cm.get_cmap('Paired', len(class_names))
    else:
        colormap = plt.cm.get_cmap('tab10', len(class_names))

    for title, embeddings in embeddings_to_plot.items():
        labels = labels_to_plot[title]
        colors = [colormap(label) for label in labels]

        umap_reducer = umap.UMAP(n_neighbors=config.UMAP.N_NEIGHBORS, min_dist=config.UMAP.MIN_DIST, n_components=2, random_state=seed)
        umap_embeddings = umap_reducer.fit_transform(embeddings)

        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colormap(class_id),
                              markersize=10, label=class_names[class_id]) for class_id in classes]

        fig, ax = plt.subplots(figsize=(7, 6))
        scatter = ax.scatter(umap_embeddings[:, 0], umap_embeddings[:, 1], c=colors, s=20)
        ax.set_title(f"{title}")
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")
        ax.legend(handles=handles, title='Patient type', loc='best', edgecolor='black')

        save_path_pdf = save_path.format(title=title, format='pdf')
        save_path_svg = save_path.format(title=title, format='svg')
        plt.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
        plt.savefig(save_path_svg, format='svg', bbox_inches='tight')


def calculate_cluster_metrics(embeddings, labels):
    results = {}
    for key in embeddings:
        silhouette = silhouette_score(embeddings[key], labels[key])
        davies_bouldin = davies_bouldin_score(embeddings[key], labels[key])
        results[key] = {
            "Silhouette Score": silhouette,
            "Davies-Bouldin Index": davies_bouldin
        }
    return results


# -----------------------------------------------------------------
# OPTIMIZED ABLATION UTILS (VECTORIZED)
# -----------------------------------------------------------------
def generate_training_confidences(batch_size, num_modalities, device='cuda'):
    """Generates a tensor of random confidences ~ U(0.2, 0.9)."""
    min_val, max_val = 0.2, 0.9
    return (max_val - min_val) * torch.rand(batch_size, num_modalities, device=device) + min_val


def stress_test_a_perturbation(confidences, drop_ratio=0.2, drop_val=0.1):
    """Stress Test A: Drops the confidence score to 0.1 for 20% of patients."""
    perturbed_conf = confidences.clone()
    batch_size, num_modalities = perturbed_conf.shape
    num_to_perturb = int(batch_size * drop_ratio)

    if num_to_perturb == 0:
        return perturbed_conf

    patient_indices = torch.randperm(batch_size, device=confidences.device)[:num_to_perturb]
    modality_indices = torch.randint(0, num_modalities, (num_to_perturb,), device=confidences.device)

    perturbed_conf[patient_indices, modality_indices] = drop_val
    return perturbed_conf


def stress_test_b_perturbation(embeddings, confidences, drop_ratio=0.2, high_conf_val=0.9):
    """
    Stress Test B: Replaces one modality's embedding with Gaussian noise
    and falsely forces its confidence to 0.9. (Fully Vectorized)
    """
    perturbed_emb = embeddings.clone()
    perturbed_conf = confidences.clone()

    batch_size, num_modalities, embed_dim = perturbed_emb.shape
    num_to_perturb = int(batch_size * drop_ratio)

    if num_to_perturb == 0:
        return perturbed_emb, perturbed_conf

    patient_indices = torch.randperm(batch_size, device=embeddings.device)[:num_to_perturb]
    modality_indices = torch.randint(0, num_modalities, (num_to_perturb,), device=embeddings.device)

    # Vectorized assignment
    perturbed_emb[patient_indices, modality_indices] = torch.randn(num_to_perturb, embed_dim, device=embeddings.device)
    perturbed_conf[patient_indices, modality_indices] = high_conf_val

    return perturbed_emb, perturbed_conf